This notebook tests basic simulation by recapitulating a simulated dataset several times and measuring variance.

Imports

In [1]:
import sys
import os
package_path = os.path.abspath("..")
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-05-28 19:29:06.760472: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-28 19:29:06.765305: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-28 19:29:06.765318: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
#dask imports
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

Make the dask cluster & client in accordance with resource avail and model size

In [3]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=2:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)

Ship the package to the workers:

(This is again a dev hack which will not be required after code is packaged)

In [4]:
cluster.scale(jobs=5)

In [5]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [6]:
#let's make an ortho object out of the simulated data
test=scm.ortho()
test.criss_cross(client=client,dat=dat)
test.extract_params(client)

In [7]:
description_primordial=scm.describe_parameters(client,parameters=test.by_cre_parameters.result(),dat=dat,split="cre_id")

In [8]:
description_primordial

cells  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[1] C(rep_id)[2] C(rep_id)[3] cre_id             
1                   0                   0            1            0            nobody       503   
                                                     0            1            nobody       500   
                                        1            0            0            nobody       499   
0                   1                   0            1            0            nobody       443   
                                                     0            1            nobody       441   
                                        1            0            0            nobody       440   
1                   0                   0            1            0            somebody     522   
                                        1            0            0            somebody     504   
                                        0            0            1            somebody     498   
0                   1                   1            0            0            somebody     464   
                                        0            0            1            somebody     457   
                                                     1            0            somebody     434   
1                   0                   0            0            1            everybody    514   
                                        1            0            0            everybody    501   
                                        0            1            0            everybody    499   
0                   1                   0            1            0            everybody    465   
                                                     0            1            everybody    455   
                                        1            0            0            everybody    451   
1                   0                   0            0            1            redgene      523   
                                        1            0            0            redgene      495   
                                        0            1            0            redgene      493   
0                   1                   0            1            0            redgene      465   
                                                     0            1            redgene      445   
                                        1            0            0            redgene      443   
1                   0                   1            0            0            neurogene    499   
                                        0            1            0            neurogene    491   
                                                     0            1            neurogene    490   
0                   1                   1            0            0            neurogene    465   
                                        0            1            0            neurogene    460   
                                                     0            1            neurogene    453   

                                                                                                  nb  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[1] C(rep_id)[2] C(rep_id)[3] cre_id                  
1                   0                   0            1            0            nobody       1.769176   
                                                     0            1            nobody       1.769176   
                                        1            0            0            nobody       1.769176   
0                   1                   0            1            0            nobody       0.927057   
                                                     0            1            nobody       0.927057   
                                        1            0            0            nobody       0.927057   
1                   0                   0            1            0            somebody  

In [10]:
description_primordial=scm.auto_partition(description_primordial,50)

In [38]:
description_primordial.compute()

,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p
0,1,0,0,1,0,nobody,503,1.771581,0.454803,0.772871,2.165975,3.220582,0.550081
1,1,0,0,0,1,nobody,500,1.771581,0.880360,0.772871,2.165975,3.220582,0.550081
2,1,0,1,0,0,nobody,499,1.771581,0.772829,0.772871,2.165975,3.220582,0.550081
3,0,1,0,1,0,nobody,443,0.928642,0.454803,0.772871,2.165975,1.326788,0.699917
4,0,1,0,0,1,nobody,441,0.928642,0.880360,0.772871,2.165975,1.326788,0.699917
5,0,1,1,0,0,nobody,440,0.928642,0.772829,0.772871,2.165975,1.326788,0.699917
6,1,0,0,1,0,somebody,522,12.479561,0.519388,1.110870,3.037000,63.760241,0.195726
7,1,0,1,0,0,somebody,504,12.479561,0.799652,1.110870,3.037000,63.760241,0.195726
8,1,0,0,0,1,somebody,498,12.479561,0.902683,1.110870,3.037000,63.760241,0.195726
9,0,1,1,0,0,somebody,464,9.969256,0.799652,1.110870,3.037000,42.694330,0.233503


In [134]:
x=scm.simulate_from_description(description_primordial)
x=x.compute()
x=scm.undo_one_hot_encoding(x)
x=x.rename({'zinb_sample':'umis_mpra_bc'},axis=1)[['rep_id','cre_id','cell_type','umis_mpra_bc']].reset_index()
x

,index,rep_id,cre_id,cell_type,umis_mpra_bc
0,2,2,nobody,blood,0
1,3,2,nobody,blood,0
2,7,2,nobody,blood,0
3,12,2,nobody,blood,0
4,13,2,nobody,blood,5
...,...,...,...,...,...
14307,7084,3,neurogene,brain,92
14308,7085,3,neurogene,brain,43
14309,7089,3,neurogene,brain,35
14310,7091,3,neurogene,brain,125


In [127]:
recap=scm.ortho()
recap.criss_cross(client=client,dat=x)
recap.extract_params(client)

In [128]:
description_x=scm.describe_parameters(client,parameters=recap.by_cre_parameters.result(),dat=x,split="cre_id")

In [132]:
working=scm.undo_one_hot_encoding(description_x.reset_index())

#zi=working.drop(columns=['cre_id','cells','nb','theta','p','r','sigmasquare']).drop_duplicates()#.groupby("rep_id").agg('mean')# zi
#zi
working.drop(columns=['rep_id','cells','zi']).drop_duplicates()# nb

,cre_id,nb,theta,r,sigmasquare,p,cell_type
0,nobody,1.796245,0.828418,2.289694,3.205383,0.560384,blood
3,nobody,0.952848,0.828418,2.289694,1.349373,0.706142,brain
6,somebody,12.648378,1.132906,3.104664,64.177771,0.197083,blood
9,somebody,10.019911,1.132906,3.104664,42.357908,0.236554,brain
12,everybody,103.992330,1.248328,3.484512,3207.555687,0.032421,blood
15,everybody,102.803918,1.248328,3.484512,3135.838272,0.032784,brain
18,redgene,107.753354,1.295296,3.652076,3286.982177,0.032782,blood
21,redgene,31.218540,1.295296,3.652076,298.079697,0.104732,brain
24,neurogene,15.556513,1.375947,3.958822,76.687092,0.202857,blood
27,neurogene,95.786237,1.375947,3.958822,2413.395363,0.039689,brain


In [135]:
cluster.close()